# Audit preprocessing dan duplikasi lima dataset
## Ringkasan
Notebook ini memeriksa satuan data: **satu baris = satu molekul berdasarkan canonical isomeric SMILES**.
Label aroma yang sama pada molekul berbeda bukan duplikasi.

Seluruh pemeriksaan membaca snapshot lokal dan menghitung ulang di memori. Tidak menulis ulang dataset,
tidak menyaring bahan berdasarkan penggunaan parfum, tidak melatih model, dan tidak menilai performa data uji.
Rekomendasi akhir ditulis setelah hasil audit diperiksa.

## Konteks dan asumsi
- Dataset aktif: perfume-five-v1; sumber dan commit mengikuti manifest lokal.
- Label mengikuti anotasi sumber, bukan hasil pengukuran sensoris baru.
- Canonical SMILES mengatasi variasi penulisan struktur yang sama; tidak otomatis menyatukan tautomer,
  perbedaan protonasi, atau stereokimia yang tidak lengkap.
- Fitur yang sama tidak membuktikan molekul identik. Akan tetapi, fitur identik lintas split membatasi
  klaim evaluasi pada struktur baru.

Referensi: [RDKit: canonical SMILES](https://www.rdkit.org/docs/GettingStartedInPython.html#writing-molecules),
[DataSAIL: pemisahan data berdasarkan kemiripan](https://www.nature.com/articles/s41467-025-58606-8).


### Hasil audit snapshot aktif
- 13.337 baris sumber; 11.777 baris diterima; setelah penggabungan tersisa 6.686 molekul.
- 436 molekul memiliki beberapa penulisan SMILES input yang sudah berhasil disatukan.
- Duplikat canonical SMILES dan overlap canonical train-test: 0.
- Morgan saja: 367/1.368 molekul uji memiliki fitur identik dengan data latih (26,8%).
- Morgan + 5: 260/1.368 molekul uji memiliki fitur identik dengan data latih (19,0%).
- Seluruh 545 fold yang diperiksa memiliki overlap fitur Morgan + 5 antara train dan validation.
- 7/7 blok kode audit berhasil. Data dan kode alignment tetap sama.
- Status evaluasi: **perlu meninjau split berbasis kelompok sebelum eksperimen final**.
  Audit ini tidak mengukur apakah atau sebesar apa skor model menjadi optimistis.


## 1. Baca data dan periksa checksum
Jalankan dengan environment proyek, dari root repository atau folder notebooks.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from rdkit import Chem, rdBase

ROOT = Path.cwd().resolve()
if not (ROOT / "alignment/perfume_config.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "alignment/perfume_config.json").is_file(), "Jalankan dari repository atau notebooks."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from alignment.perfume_data import merge_sources, extract_features
from src.build_dataset import _attach_smiles, _attach_sigma, _cas_bridge

DATA = ROOT / "data/builds/perfume-five-v1"
read_json = lambda p: json.loads(p.read_text(encoding="utf-8"))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
manifest = read_json(DATA / "dataset_manifest.json")
config = read_json(DATA / "config.json")
split = read_json(DATA / "splits.json")
source_audit = read_json(DATA / "audit.json")
saved_source = read_json(DATA / "sources.json")
records = pd.read_csv(DATA / "records.csv")
for col in ("labels", "sources"):
    records[col] = records[col].map(json.loads)
provenance = read_json(DATA / "provenance.json")
assert rdBase.rdkitVersion == manifest["feature_spec"]["rdkit_version"]
assert config == read_json(ROOT / "alignment/perfume_config.json")
protected_paths = list(DATA.glob("*")) + list((ROOT / "alignment").glob("*.py"))
before_hashes = {str(p): sha(p) for p in protected_paths if p.is_file()}
for name, expected in manifest["files"].items():
    assert sha(DATA / name) == expected, name
train, test = set(split["train"]), set(split["test"])
assert not train & test
assert train | test == set(range(len(records)))
print("Commit sumber:", config["revision"])
print("RDKit:", rdBase.rdkitVersion)
print("Checksum dataset: PASS")
display(pd.DataFrame([{"molekul":len(records), "latih":len(train),
                       "uji":len(test), "label_model":len(manifest["labels"])}]))


Commit sumber: 8054ea98ed675005ec10e67359902f500e4911b0
RDKit: 2026.03.4
Checksum dataset: PASS


,molekul,latih,uji,label_model
0,6686,5318,1368,109


## 2. Ulangi penggabungan sumber di memori
Penghubung Stimulus–CID–SMILES dan CAS diperiksa oleh fungsi produksi.
Baris yang ditolak tetap dapat ditelusuri pada rejected_records.json.
Ini adalah pembersihan yang sudah berlaku, bukan penyaringan tambahan khusus parfum.

In [2]:
cache = ROOT / "data/snapshots/perfume-five-v1" / config["revision"]
tables = {}
for source, kinds in config["sources"].items():
    for kind, relative in kinds.items():
        path = cache / relative
        assert sha(path) == saved_source[relative]["sha256"], relative
        tables[f"{source}_{kind}"] = pd.read_csv(path, dtype=str)
rebuilt, taxonomy, audit, rejected, rebuilt_provenance, unmapped, mappings = merge_sources(tables, config)
assert rebuilt.to_dict("records") == records.to_dict("records")
assert rebuilt_provenance == provenance
assert rejected == read_json(DATA / "rejected_records.json")
assert audit["sources"] == source_audit["sources"]
assert not records["smiles"].duplicated().any()
assert records["source_row"].tolist() == list(range(len(records)))
assert set(records.iloc[sorted(train)]["smiles"]).isdisjoint(records.iloc[sorted(test)]["smiles"])
source_counts = pd.DataFrame(audit["sources"]).T
display(source_counts)
print("Penggabungan ulang identik dengan dataset tersimpan: PASS")
print("Duplikat canonical SMILES: 0; overlap canonical latih-uji: 0")


Penggabungan ulang identik dengan dataset tersimpan: PASS
Duplikat canonical SMILES: 0; overlap canonical latih-uji: 0


,rows,missing_identity,invalid_smiles,multifragment,empty_labels,unmapped_labels,invalid_behavior,accepted
goodscents,4626,0,0,183,4,335,0,4104
ifra,1146,0,0,28,0,0,0,1118
leffingwell,3522,0,0,0,0,114,0,3408
arctander,3102,278,0,102,209,73,0,2440
sigma,941,204,0,0,0,30,0,707


## 3. Molekul yang berulang antarsumber
Molekul yang sama digabung menjadi satu baris. Label positif dari sumber berbeda dikumpulkan.
Perbedaan daftar label berarti anotasi berbeda, bukan otomatis label salah: label yang tidak
dicatat tidak membuktikan aroma tersebut tidak ada.

In [3]:
by_molecule = defaultdict(list)
for row in provenance:
    by_molecule[row["smiles"]].append(row)
summary = {
    "raw_source_rows": int(source_counts["rows"].sum()),
    "accepted_source_rows": len(provenance),
    "unique_canonical_molecules": len(records),
    "consolidated_extra_rows": len(provenance) - len(records),
    "repeated_molecule_groups": sum(len(rows)>1 for rows in by_molecule.values()),
    "multisource_molecules": sum(len({r["source"] for r in rows})>1 for rows in by_molecule.values()),
    "molecules_with_different_source_annotations": sum(
        len({tuple(r["labels"]) for r in rows})>1 for rows in by_molecule.values()),
    "canonical_duplicate_rows": int(records["smiles"].duplicated().sum()),
    "canonical_train_test_overlap": 0
}
for row in records.itertuples():
    origin = by_molecule[row.smiles]
    assert set(row.labels) == {label for item in origin for label in item["labels"]}
display(pd.DataFrame(summary.items(), columns=["pemeriksaan", "jumlah"]))
source_sets = {source:set(records.loc[records.sources.map(lambda s:source in s),"smiles"])
               for source in config["sources"]}
overlap = pd.DataFrame({b:{a:len(source_sets[a]&source_sets[b]) for a in source_sets}
                       for b in source_sets})
print("Matriks molekul bersama (diagonal = molekul unik setiap sumber):")
display(overlap)
label_counts = Counter(label for row in records.labels for label in row)
display(pd.DataFrame(label_counts.most_common(10), columns=["label", "molekul_unik"]))
same_label_examples = records[records.labels.map(lambda x:"sweet" in x)].copy()
same_label_examples["length"] = same_label_examples.smiles.str.len()
print("Contoh struktur berbeda yang sama-sama berlabel sweet:")
display(same_label_examples.sort_values(["length","smiles"])[["smiles","labels","sources"]].head(4))


Matriks molekul bersama (diagonal = molekul unik setiap sumber):
Contoh struktur berbeda yang sama-sama berlabel sweet:


,pemeriksaan,jumlah
0,raw_source_rows,13337
1,accepted_source_rows,11777
2,unique_canonical_molecules,6686
3,consolidated_extra_rows,5091
4,repeated_molecule_groups,2838
5,multisource_molecules,2808
6,molecules_with_different_source_annotations,2746
7,canonical_duplicate_rows,0
8,canonical_train_test_overlap,0


,goodscents,ifra,leffingwell,arctander,sigma
goodscents,4050,802,2093,1112,657
ifra,802,1047,568,376,249
leffingwell,2093,568,3408,1030,637
arctander,1112,376,1030,2383,384
sigma,657,249,637,384,707


,label,molekul_unik
0,fruity,2498
1,green,2069
2,floral,1811
3,sweet,1595
4,herbal,1271
5,woody,1068
6,fatty,764
7,oily,691
8,spicy,637
9,fresh,632


,smiles,labels,sources
4997,CCN,"[cheesy, grape, sweet]","[leffingwell, sigma]"
5005,CCO,"[ethereal, medicinal, sweet]","[arctander, goodscents, leffingwell]"
5867,CSC,"[berry, corn, creamy, fruity, green, onion, pu...","[arctander, goodscents, leffingwell, sigma]"
4889,CCCO,"[apple, fermented, fruity, musty, peanut, pear...","[goodscents, ifra, leffingwell, sigma]"


## 4. Apakah penulisan SMILES mentah bisa berbeda?
Audit hanya menghitung baris yang diterima, kemudian membandingkan penulisan asli dengan kunci canonical.
Sigma memakai penghubung CAS sehingga tidak semua sumber menyediakan string asli secara langsung.
Perbedaan teks saja tidak cukup untuk menyatakan dua molekul berbeda.

In [4]:
frames = {source:_attach_smiles(tables[f"{source}_behavior"],
                                   tables[f"{source}_stimuli"],tables[f"{source}_molecules"])
          for source in config["sources"] if source != "sigma"}
cas_map, ambiguous = _cas_bridge(tables, frames)
frames["sigma"], _ = _attach_sigma(tables["sigma_behavior"], tables["sigma_stimuli"], cas_map)
spellings = defaultdict(set)
for row in provenance:
    raw = frames[row["source"]].iloc[row["row"]]["__smiles__"]
    canonical = Chem.MolToSmiles(Chem.MolFromSmiles(raw), isomericSmiles=True)
    assert canonical == row["smiles"]
    spellings[canonical].add(raw)
variants = [(s, sorted(v)) for s,v in spellings.items() if len(v)>1]
summary["same_canonical_multiple_input_spellings"] = len(variants)
print("Molekul dengan beberapa penulisan SMILES input:", len(variants))
display(pd.DataFrame(variants[:3],columns=["canonical_smiles","penulisan_input"]))
example = [Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in ["CCO","OCC"]]
assert example[0] == example[1]
print("Contoh ilustrasi etanol: CCO dan OCC menjadi", example[0])


Molekul dengan beberapa penulisan SMILES input: 436
Contoh ilustrasi etanol: CCO dan OCC menjadi CCO


,canonical_smiles,penulisan_input
0,COc1ccc(C(C)=O)cc1,"[CC(=O)C1=CC=C(C=C1)OC, COc1ccc(C(C)=O)cc1]"
1,C=Cc1ccccc1,"[C=CC1=CC=CC=C1, C=Cc1ccccc1]"
2,OCc1ccccc1,"[C1=CC=C(C=C1)CO, OCc1ccccc1]"


## 5. Periksa fitur yang identik
Fitur dihitung ulang dari struktur dengan fungsi produksi. Tidak membuka test.npz.
Perbandingan dilakukan untuk Morgan saja (A/C) dan Morgan + 5 deskriptor (B/D).
Pemeriksaan ini tidak menggunakan skor model atau memilih hiperparameter.

In [5]:
morgan, descriptors = extract_features(records, config)
assert np.isfinite(descriptors).all()
with np.load(DATA / "train.npz", allow_pickle=False) as stored:
    indices = stored["row_index"]
    assert indices.tolist() == split["train"]
    assert np.array_equal(stored["morgan"], morgan[indices])
    assert np.array_equal(stored["descriptors"], descriptors[indices])
    training_truth = stored["truth"].copy()
    expected = np.array([[int(label in records.iloc[i].labels) for label in manifest["labels"]]
                         for i in indices], dtype=np.uint8)
    assert np.array_equal(training_truth, expected)
positions = {int(global_i):local_i for local_i,global_i in enumerate(indices)}
training_positive = training_truth.sum(axis=0)
assert (training_positive>=config["labels"]["min_train_positive"]).all()
summary["all_zero_training_target_rows"] = int((training_truth.sum(axis=1)==0).sum())

connectivity_groups = defaultdict(list)
morgan_groups, full_groups = defaultdict(list), defaultdict(list)
for i, smiles in enumerate(records.smiles):
    mol = Chem.MolFromSmiles(smiles)
    Chem.RemoveStereochemistry(mol)
    connectivity_groups[Chem.MolToSmiles(mol,isomericSmiles=True)].append(i)
    morgan_groups[morgan[i].tobytes()].append(i)
    full_groups[(morgan[i].tobytes(),descriptors[i].tobytes())].append(i)

def profile(groups):
    duplicated = [v for v in groups.values() if len(v)>1]
    crossing = [v for v in duplicated if set(v)&train and set(v)&test]
    return {"duplicate_groups":len(duplicated),
            "rows_in_duplicate_groups":sum(map(len,duplicated)),
            "cross_split_groups":len(crossing),
            "test_rows_matching_training":sum(len(set(v)&test) for v in crossing)}
feature_profile = pd.DataFrame({"Morgan":profile(morgan_groups),
                               "Morgan_plus_5":profile(full_groups)}).T
display(feature_profile)
summary["feature_overlap"] = feature_profile.to_dict("index")
summary["structure_groups_ignoring_stereochemistry"] = sum(len(v)>1 for v in connectivity_groups.values())
full_cross = [v for v in full_groups.values() if set(v)&train and set(v)&test]
examples = []
for group_number, rows in enumerate(full_cross[:3],1):
    for i in rows:
        examples.append({"group":group_number,"row":i,"split":"train" if i in train else "test",
                         "smiles":records.iloc[i].smiles,"sources":records.iloc[i].sources})
display(pd.DataFrame(examples))


,duplicate_groups,rows_in_duplicate_groups,cross_split_groups,test_rows_matching_training
Morgan,724,2047,300,367
Morgan_plus_5,638,1520,238,260


,group,row,split,smiles,sources
0,1,19,train,C/C(=C/CCC1(C)C2CC3C(C2)C31C)CO,[leffingwell]
1,1,20,test,C/C(=C/CC[C@]1(C)C2C[C@@H]3[C@H](C2)C31C)CO,[ifra]
2,1,25,train,C/C(=C\CCC1(C)C2CC3C(C2)C31C)CO,[goodscents]
3,1,929,test,CC(=CCCC1(C)C2CC3C(C2)C31C)CO,[arctander]
4,2,22,test,C/C(=C\C(C)C)C(=O)O,"[goodscents, leffingwell]"
5,2,918,train,CC(=CC(C)C)C(=O)O,[goodscents]
6,3,24,train,C/C(=C\CC(C)C=O)C(C)C,[goodscents]
7,3,928,test,CC(=CCC(C)C=O)C(C)C,[ifra]


## 6. Periksa fold validasi silang
Selain train–test, kelompok fitur identik juga diperiksa pada train–validation di setiap label.
Pemeriksaan kelas memakai label data latih saja. Tidak mengevaluasi model.

In [6]:
group_id = {}
for group_number, rows in enumerate(full_groups.values()):
    for i in rows:
        group_id[i] = group_number
fold_profiles = []
for j,label in enumerate(manifest["labels"]):
    seen = []
    for k,fold in enumerate(split["folds"][label],1):
        a,b = set(fold["train"]),set(fold["validation"])
        assert not a & b and a | b == train
        for partition in (a,b):
            target = training_truth[[positions[i] for i in partition],j]
            assert set(np.unique(target)) == {0,1}
        seen.extend(b)
        training_groups = {group_id[i] for i in a}
        hits = sum(group_id[i] in training_groups for i in b)
        fold_profiles.append({"label":label,"fold":k,"validation_rows":len(b),
                              "validation_rows_matching_training_features":hits})
    assert sorted(seen) == sorted(train)
fold_table = pd.DataFrame(fold_profiles)
summary["cv_folds_checked"] = len(fold_table)
summary["cv_folds_with_feature_overlap"] = int(
    (fold_table.validation_rows_matching_training_features>0).sum())
print("Pemeriksaan indeks dan kelas seluruh fold: PASS")
display(fold_table.describe(include="number").round(2))
print("Fold dengan fitur lintas train-validation identik:",
      summary["cv_folds_with_feature_overlap"], "/", summary["cv_folds_checked"])


Pemeriksaan indeks dan kelas seluruh fold: PASS
Fold dengan fitur lintas train-validation identik: 545 / 545


,fold,validation_rows,validation_rows_matching_training_features
count,545.00,545.00,545.00
mean,3.00,1063.60,184.71
std,1.42,0.49,10.39
min,1.00,1063.00,156.00
25%,2.00,1063.00,177.00
50%,3.00,1064.00,185.00
75%,4.00,1064.00,192.00
max,5.00,1064.00,219.00


## 7. Kesimpulan pemeriksaan dan integritas
PASS di sini berarti pemeriksaan berjalan dengan benar, bukan persetujuan bahwa rancangan evaluasi sudah final.
Data sumber, dataset tersimpan, dan kode alignment harus tetap sama setelah audit.

In [7]:
assert all(sha(Path(p)) == expected for p,expected in before_hashes.items())
summary["data_and_alignment_unchanged"] = True
summary["training_started_by_audit"] = False
summary["test_npz_deserialized"] = False
summary["audit_execution"] = "PASS"
summary["evaluation_readiness"] = "REVIEW_REQUIRED_FEATURE_OVERLAP"
print(json.dumps(summary,indent=2))


{
  "raw_source_rows": 13337,
  "accepted_source_rows": 11777,
  "unique_canonical_molecules": 6686,
  "consolidated_extra_rows": 5091,
  "repeated_molecule_groups": 2838,
  "multisource_molecules": 2808,
  "molecules_with_different_source_annotations": 2746,
  "canonical_duplicate_rows": 0,
  "canonical_train_test_overlap": 0,
  "same_canonical_multiple_input_spellings": 436,
  "all_zero_training_target_rows": 12,
  "feature_overlap": {
    "Morgan": {
      "duplicate_groups": 724,
      "rows_in_duplicate_groups": 2047,
      "cross_split_groups": 300,
      "test_rows_matching_training": 367
    },
    "Morgan_plus_5": {
      "duplicate_groups": 638,
      "rows_in_duplicate_groups": 1520,
      "cross_split_groups": 238,
      "test_rows_matching_training": 260
    }
  },
  "structure_groups_ignoring_stereochemistry": 626,
  "cv_folds_checked": 545,
  "cv_folds_with_feature_overlap": 545,
  "data_and_alignment_unchanged": true,
  "training_started_by_audit": false,
  "test_npz_de

## Tindak lanjut
- Pertahankan semua molekul valid dari lima sumber; jangan menghapus molekul karena labelnya sama.
- Pertahankan penggabungan canonical SMILES dan provenance.
- Sebelum eksperimen final, susun split berbasis kelompok agar struktur yang sama tanpa stereokimia,
  serta representasi fitur yang identik, tidak tersebar antara latih/uji maupun fold validasi.
  Morgan saja juga perlu diperiksa karena kondisi A/C tidak memakai deskriptor.
- Grouping untuk split bukan penghapusan data dan bukan clustering berdasarkan label.
- Pengelompokan kemiripan atau scaffold dapat dipakai untuk evaluasi generalisasi yang lebih ketat.
  Itu keputusan protokol terpisah; ambang kemiripan tidak dipilih berdasarkan skor test.
- Fitur tanpa stereokimia tidak dapat membedakan sebagian varian struktur. Grouping mencegah
  penyebarannya lintas split, tetapi tidak memperbaiki keterbatasan representasi tersebut.
- Dataset, label, dan split aktif belum diubah oleh notebook ini. Klaim perfume-only belum diverifikasi
  per molekul; konteks penerapan ditetapkan pada parfum.
